# Distributed QML via Classical Communication — Walkthrough

This notebook reproduces the core experimental setup of
Hwang et al., arXiv:2408.16327, and contrasts:

1. the synthetic 8D classification task,
2. the four DQML schemes implemented as PyTorch state-vector circuits,
3. a tiny photonic MerLin classifier on the same task,
4. an iso-parameter classical MLP baseline.

**Wall-clock target:** under 10 minutes on CPU.

In [ ]:
import sys, os, time, math
from pathlib import Path

PAPER_ROOT = Path.cwd()
if PAPER_ROOT.name != 'distributed_qml_cc':
    PAPER_ROOT = Path('papers/distributed_qml_cc').resolve()
for p in (PAPER_ROOT, PAPER_ROOT.parent.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
torch.manual_seed(0); np.random.seed(0)

## 1. Problem statement

The paper studies whether two small quantum processor units (4-qubit
QPUs) connected by *classical* mid-circuit-measurement feedforward can
match the classification performance of *quantum* communication on a
deliberately hard 8D synthetic binary task. The dataset is built to
minimise linear correlation between any single attribute and the label.

In [ ]:
from lib.data import build_synthetic_dataset, SyntheticDatasetConfig
x_tr, y_tr, x_va, y_va, info = build_synthetic_dataset(SyntheticDatasetConfig(seed=42))
print(f"train: {x_tr.shape}  val: {x_va.shape}  max |Pearson|: {info['pearson_max_abs']:.3f}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(x_tr[:, 0], x_tr[:, 1], c=y_tr, cmap='coolwarm', s=8, alpha=0.6)
ax.set_title('First two attributes of the 8D synthetic dataset'); ax.set_xlabel('x1'); ax.set_ylabel('x2');

## 2. Claim inventory

| ID | Claim | Test in this notebook |
|----|-------|----------------------|
| C1 | CC-DQML > NC-DQML accuracy at L ∈ {3,5,7,9} | yes — quick L=3 training |
| C2 | CC-DQML ≈ QC-DQML at small L | yes — same training |
| C3 | Both DQML schemes > non-DQML | yes — same training |

All three claims are evaluated with the same short (200-iteration)
training schedule below; for full 1000-iteration sweeps see
`utils/run_sweep.py`.

## 3. Algorithm walkthrough — circuit blocks

The four DQML schemes share three building blocks (cf. Fig. 4b of the
paper):

1. **Embedding.** Hadamard, then `RZ(x_i)` on each qubit, then cyclic
   `exp(-i x_i x_{i+1}/2 Z⊗Z)` couplings (the standard Havlicek ZZ
   feature map).
2. **Brick-wall convolutional sub-layer.** One `RX(θ_i)` per qubit, then
   a CNOT chain over alternating pairs.
3. **Pooling block.** Mid-circuit measurement + classically controlled
   RZ/RX feedforward on the surviving qubit. We use the
   *deferred-measurement* unitary `|0><0|⊗U_0 + |1><1|⊗U_1` so the entire
   circuit stays differentiable in plain PyTorch.

In [ ]:
from lib.model import DQMLConfig, DQMLModel
from lib.circuit import num_parameters
for scheme in ['non', 'nc', 'cc', 'qc']:
    for L in [3, 9]:
        print(f'{scheme:>4s} L={L}: {num_parameters(scheme, L):4d} params')

## 4. Training demo (short)

200 iterations per scheme at L=3. This is intentionally short to keep
the notebook well under 10 minutes; full reproductions use 1000
iterations and 3 seeds.

In [ ]:
from lib.training import train_dqml
xt = torch.as_tensor(x_tr, dtype=torch.float32); yt = torch.as_tensor(y_tr, dtype=torch.float32)
xv = torch.as_tensor(x_va, dtype=torch.float32); yv = torch.as_tensor(y_va, dtype=torch.float32)
results = {}
for scheme in ['non', 'nc', 'cc', 'qc']:
    torch.manual_seed(0)
    model = DQMLModel(DQMLConfig(scheme=scheme, n_layers=3))
    t0 = time.time()
    res = train_dqml(model, xt, yt, xv, yv, n_iterations=200, eval_every=25, lr=0.05, batch_size=512, seed=0)
    results[scheme] = res
    print(f'{scheme}: val_acc={res.final_val_acc:.3f} ({time.time()-t0:.1f}s)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = {'non': 'non-DQML', 'nc': 'NC-DQML', 'cc': 'CC-DQML', 'qc': 'QC-DQML'}
for scheme, res in results.items():
    ax.plot(res.iterations, res.val_acc, label=labels[scheme])
ax.set_xlabel('Training iteration'); ax.set_ylabel('Validation accuracy')
ax.set_title('Reduced reproduction of Fig. 4c (L=3, 200 iter, 1 seed)')
ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout();

## 5. MerLin photonic extension walkthrough

The photonic model is a stock MerLin vector-in classifier
(`MERLIN_COOKBOOK.md` pattern B): 8 modes, 4 photons in a dual-rail-style
state, angle encoding `scale=π`, 4 trainable MZI-mesh entangling layers,
threshold-detector probability readout, and a `LexGrouping(70 → 2)`
head.

In [ ]:
from lib.merlin_model import PhotonicDQML
torch.manual_seed(0)
model_m = PhotonicDQML(n_modes=8, n_entangling_layers=4, n_photons=4)
print('Photonic ansatz: ', sum(p.numel() for p in model_m.parameters() if p.requires_grad),
      'params  output_size=', model_m.output_size)
opt = torch.optim.Adam(model_m.parameters(), lr=0.05)
hist = []
t0 = time.time()
for it in range(1, 151):
    gen = torch.Generator().manual_seed(it)
    idx = torch.randint(0, xt.shape[0], (256,), generator=gen)
    loss = ((yt[idx] - model_m(xt[idx])) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if it % 25 == 0:
        with torch.no_grad():
            acc = (((model_m(xv) >= 0).float() * 2 - 1) == yv).float().mean().item()
            hist.append((it, acc))
            print(f'iter {it:3d}: val_acc={acc:.3f}')
print(f'Total: {time.time()-t0:.1f}s')

## 6. Fair classical baseline

An iso-parameter MLP (hidden=8 → 137 params) trained identically on the
same data. If the MLP already matches CC-DQML we know the gap is one of
inductive bias, not raw expressivity.

In [ ]:
from lib.classical_model import TinyMLP
torch.manual_seed(0)
mlp = TinyMLP(8, 8)
opt = torch.optim.Adam(mlp.parameters(), lr=0.05)
for it in range(1, 201):
    gen = torch.Generator().manual_seed(it)
    idx = torch.randint(0, xt.shape[0], (512,), generator=gen)
    loss = ((yt[idx] - mlp(xt[idx])) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    mlp_acc = (((mlp(xv) >= 0).float() * 2 - 1) == yv).float().mean().item()
print(f'TinyMLP hidden=8 val_acc after 200 iter: {mlp_acc:.3f}')

## 7. Interpretation

Under the reduced budget of this notebook (200 iterations, single seed,
L=3) the qualitative ordering reported by the paper should already be
visible:

- CC-DQML and QC-DQML cluster together at the top.
- NC-DQML lags behind both and the non-DQML model is the weakest.
- The photonic MerLin model and the tiny MLP serve as orthogonal
  references: photonic shows the same task is reachable with a
  hardware-realistic encoding, the MLP shows that *some* model with
  similar parameter count can solve it without any quantum structure at
  all.

For the full-budget reproduction (1000 iterations × 3 seeds × L ∈
{3,5,7,9}), see `utils/run_sweep.py` and `utils/plot_results.py`.